# **CRAWLING BERITA DARI DETIK.COM**

In [1]:
import requests
from bs4 import BeautifulSoup
import time
import re
import string
import sys
import pandas as pd
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
import random

# --- FUNGSI-FUNGSI BANTUAN SCRAPING ---
def print_progress(kategori, current_page, total_pages):
    """Menampilkan progress bar di konsol."""
    percent = (current_page / total_pages) * 100 if total_pages > 0 else 0
    bar_length = 20
    filled_length = int(bar_length * current_page // total_pages) if total_pages > 0 else 0
    bar = '█' * filled_length + '-' * (bar_length - filled_length)
    sys.stdout.write(f'\r{kategori} - Page {current_page}/{total_pages} [{bar}] {percent:.2f}%')
    sys.stdout.flush()
    if current_page == total_pages:
        sys.stdout.write('\n\n')

def get_session():
    """Membuat sesi permintaan dengan mekanisme percobaan ulang."""
    session = requests.Session()
    retry_strategy = Retry(
        total=5,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["GET"],
        backoff_factor=1
    )
    adapter = HTTPAdapter(max_retries=retry_strategy)
    session.mount("https://", adapter)
    return session

def get_article_content_and_title(session, url):
    """Mengambil isi artikel dan judul dari URL."""
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/108.0.0.0 Safari/537.36'
    }
    try:
        r = session.get(url, headers=headers, timeout=15)
        r.raise_for_status()
        soup = BeautifulSoup(r.text, "html.parser")

        title = get_article_title(soup)
        content_selectors = [
            "div.detail-konten", "div.news-detail__content", "div.itp_bodycontent",
            "div.content-text", "div.article-content", "div.text_area"
        ]

        paragraphs = []
        for selector in content_selectors:
            content_divs = soup.select(selector)
            if content_divs:
                for div in content_divs:
                    for p in div.find_all("p"):
                        text = p.get_text(strip=True)
                        if text and not text.lower().startswith("baca juga"):
                            paragraphs.append(text)
                if paragraphs:
                    break
        
        if not paragraphs:
            body_text = soup.find("article")
            if body_text:
                for p in body_text.find_all("p"):
                    text = p.get_text(strip=True)
                    if text and not text.lower().startswith("baca juga"):
                        paragraphs.append(text)

        content = " ".join(paragraphs)
        return title, content
    except requests.exceptions.RequestException as e:
        print(f"Error fetching {url}: {e}", file=sys.stderr)
        return "Judul Tidak Ditemukan", ""

def get_article_title(soup):
    """Mengambil judul artikel dari berbagai kemungkinan lokasi."""
    title_tag = soup.find("h1", class_="detail-title")
    if title_tag:
        return title_tag.get_text(strip=True)
    
    title_tag = soup.find("h2", class_="media__title")
    if title_tag:
        return title_tag.get_text(strip=True)
        
    title_tag = soup.find("title")
    if title_tag:
        return title_tag.get_text(strip=True).replace(" - detiknews", "").replace(" - detikfinance", "")

    return "Judul Tidak Ditemukan"

def extract_id(url):
    """
    Ekstrak ID berita dari URL, mengatasi format yang berbeda.
    """
    id_match_d = re.search(r"/d-(\d+)", url)
    if id_match_d:
        return id_match_d.group(1)
    id_match_end = re.search(r"-(\d+)$", url)
    if id_match_end:
        return id_match_end.group(1)
    id_match_middle = re.search(r"(\d+)\.html$", url)
    if id_match_middle:
        return id_match_middle.group(1)
        
    return None

# --- FUNGSI UTAMA SCRAPING ---
def berita(categories, pages_per_category=10):
    """Fungsi utama untuk melakukan crawling berita dan menyimpan hasilnya."""
    start_time = time.time()
    session = get_session()
    all_articles_data = []
    processed_links = set()

    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/108.0.0.0 Safari/537.36'
    }

    base_urls = {
        "politik": "https://news.detik.com/indeks/berita/",
        "hukum": "https://news.detik.com/indeks/berita/",
        "ekonomi": "https://finance.detik.com/indeks/",
        "detikx": "https://news.detik.com/x/indeks/",
        "hiburan": "https://hot.detik.com/indeks/",
        "internasional": "https://news.detik.com/indeks/berita/",
        "sepakbola": "https://sport.detik.com/sepakbola/indeks/",
        "olahraga": "https://sport.detik.com/indeks/",
        "lingkungan": "https://www.detik.com/tag/lingkungan",
        "otomotif": "https://oto.detik.com/indeks"
    }
    
    categories = list(set(categories))

    for cat in categories:
        current_url = base_urls.get(cat.lower(), f"https://{cat.lower()}.detik.com/indeks/")
        print(f"--- Memulai crawling untuk kategori: {cat} ---")
        
        for page_count in range(1, pages_per_category + 1):
            url = f"{current_url}?page={page_count}"
            if cat.lower() == "lingkungan":
                url = f"https://www.detik.com/tag/lingkungan?page={page_count}"
            
            print_progress(cat, page_count, pages_per_category)

            try:
                r = session.get(url, headers=headers, timeout=15)
                r.raise_for_status()
                soup = BeautifulSoup(r.text, "html.parser")
                article_links = soup.select("a.media__link")
                
                for a in article_links:
                    link = a["href"]
                    if link in processed_links:
                        continue
                    processed_links.add(link)
                    berita_id = extract_id(link)
                    title, content = get_article_content_and_title(session, link)
                    
                    if content:
                        print(f"\n--- Data {len(all_articles_data) + 1} ---")
                        print(f"ID Berita: {berita_id}")
                        print(f"Judul: {title}")
                        print(f"Abstrak (Raw): {content}")
                        print(f"Kategori: {cat}")
                        print("-------------------------------------------\n")

                        all_articles_data.append({
                            "id_berita": berita_id,
                            "judul_berita": title,
                            "isi_berita_original": content,
                            "kategori_berita": cat
                        })
                    time.sleep(random.uniform(1, 3))
            except requests.exceptions.RequestException as e:
                print(f"\n❌ Gagal mengakses {url}: {e}", file=sys.stderr)
                break
    
    df = pd.DataFrame(all_articles_data)
    df.to_csv("crawling_detik_berita.csv", index=False, encoding="utf-8-sig")

    end_time = time.time()
    elapsed = int(end_time - start_time)
    jam, sisa = divmod(elapsed, 3600)
    menit, detik = divmod(sisa, 60)

    print("\n✅ Seluruh data berhasil dikumpulkan!")
    print(f"📊 Total entri: {len(df)}")
    print(f"⏱️ Waktu eksekusi: {jam} jam {menit} menit {detik} detik")
    
    print("\nBerikut adalah 5 entri pertama yang berhasil dikumpulkan:")
    print(df.head())

    return df

if __name__ == '__main__':
    categories = ["politik", "hukum", "ekonomi", "lingkungan", "hiburan", "internasional", "otomotif", "olahraga", "sepakbola"]
    berita(categories, pages_per_category=5)

--- Memulai crawling untuk kategori: politik ---
politik - Page 1/5 [████----------------] 20.00%


--- Data 1 ---
ID Berita: 8657143
Judul: Kemenkes Ungkap Korban Karhutla di 6 Pronvisi: 4 Meninggal, 347 Luka-luka
Abstrak (Raw): Kementerian Kesehatan (Kemenkes) menyampaikan informasi terdata mengenai korban kebakaran hutan dan lahan (karhutla) di 6 provinsi Indonesia. Total ada 347 orang terluka dan 4 orang meninggal dunia. Data korban karhutla disampaikan oleh Jubir Kemenkes Widyawati lewat YouTube BNPB, dilihat detikcom, Kamis (10/9/2026). Ia menyampaikan data korban di 6 provinsi terdampak karhutla. "Kita tahu bahwa saat ini seperti diterangkan Pak Berton ada 3 bencana kita alami, ini adalah data bencana masyarakat terdampak karhutla per 9 September 2026 terdampat di Kalsel, Kalteng, Kalbar, Kaltim, Jambi, Riau, dan Sumsel," kata Widyawati. SCROLL TO CONTINUE WITH CONTENT Ia lalu menyampaikan rincian korban di 6 provinsi tersebut. Korban yang paling banyak terdapat di Kalbar. "Kota Banjarbaru ada beberapa yang alami luka berat dan rawat inap, kemdian luka ringan dan rawat jalan,


--- Data 2 ---
ID Berita: 8657138
Judul: Sejoli Tega Bunuh Bayi Baru Lahir di Jakbar Terancam 15 Tahun Bui
Abstrak (Raw): Pasangan kekasih pria berinisial R (28) dan wanita E (23) ditangkap polisi karena diduga membunuh bayi mereka di Tamansari, Jakarta Barat (Jakbar). Sejoli itu dijerat pasal berlapis. Kepolsek Metro TamansariKompol Bobby M Zulfikar menyebut kedua tersangka dikenakan Pasal 460 KUHP Baru tentang tindak pidana pembunuhan anak sendiri oleh ibu kandung, serta Pasal 80 Undang-Undang (UU) Perlindungan Anak. Pasangan ini terancam hukuman penjara maksimal 15 tahun. "Pasal 460 KUHP baru, dan Pasal 80 UU Perlindungan Anak," kata Bobby dilansir Antara, Kamis (10/9/2026). SCROLL TO CONTINUE WITH CONTENT Kepada polisi, kedua pelaku mengaku tega membunuh bayi mereka yang baru lahir karena takut suara tangisannya terdengar oleh tetangga. Keduanya menutup saluran nafas bayi hingga korban tewas. "Itu yang bersangkutan kaget dan tidak mau mendengar anaknya yang lahir menangis. Kemudia


--- Data 3 ---
ID Berita: 8657136
Judul: KNKT Usul Sistem Hidran di Kapal Feri Dirancang Ulang Lewat Atas
Abstrak (Raw): Komite Nasional Keselamatan Transportasi (KNKT) membeberkan kendala krusial dalam penanganan tanggap darurat kebakaran di atas kapal penyeberangan. KNKT menyoroti tingginya tingkat kesulitan kru kapal saat membentang selang pemadam saat kondisi dek kendaraan padat. Ketua KNKT Soerjanto Tjahjono mengatakan, kobaran api di dek kendaraan biasanya menjalar sangat cepat dan sulit dikendalikan. Situasi ini mempersulit petugas lantaran jajaran kendaraan yang terparkir rapat kerap memblokir akses kru untuk menggelar sistem hidran. "Kalau deknya kosong gampang. Tapi begitu ada kendaraan, ternyata dari beberapa kejadian kru tidak bisa menggelar selang dari hidran untuk pemadamnya," ujar Soerjanto usai mengamati simulasi tanggap darurat kapal di KMP Jemla, Kamis (10/9/2026). SCROLL TO CONTINUE WITH CONTENT Melihat risiko besar tersebut, KNKT mengajak semua pihak regulator, Bir


--- Data 4 ---
ID Berita: 8657117
Judul: Mensos Ajak SDM Sekolah Rakyat Jadi Agen Perlinsos
Abstrak (Raw): Menteri Sosial Saifullah Yusuf atau Gus Ipul mengajak seluruh SDM di Sekolah Rakyat untuk ikut terlibat dalam Gerakan Nasional Peduli Tetangga, dengan mendaftarkan diri menjadi Agen Perlinsos (Perlindungan Sosial). Pernyataan tersebut disampaikan Gus Ipul dalam Kegiatan Pelatihan Monitoring dan Evaluasi Kinerja serta Pengelolaan Konflik dan Psychological Safety bagi SDM Sekolah Rakyat di Ballroom Hotel Grand Serpong, Tangerang, Banten, Rabu (9/9/2026). Gerakan Nasional Peduli Tetangga merupakan sebuah gerakan kepedulian sosial untuk membantu tetangga atau keluarga yang membutuhkan bantuan sosial (bansos) namun belum terdaftar, melalui pendaftaran bansos secara digital. Menurutnya, hal ini penting karena tidak semua masyarakat memiliki telepon pintar, literasi digital, dokumen lengkap, atau keberanian untuk mendaftar. SCROLL TO CONTINUE WITH CONTENT "Saya mengajak Bapak-Ibu sekali

...


--- Data 6 ---
ID Berita: 8657108
Judul: Walkot Pekanbaru Agung Nugroho Raih 4 Penghargaan IMT-GT Green City Award
Abstrak (Raw): Kepemimpinan Wali Kota Pekanbaru Agung Nugroho dalam membawa Pekanbaru menuju Metropolitan Green City memperoleh pengakuan internasional. Agung Nugroho meraih empat penghargaan dalam ajang perdana IMT-GT Green City Award 2026. Penghargaan tersebut menjadi bagian dari rangkaian pertemuan tingkat tinggi Indonesia-Malaysia-Thailand Growth Triangle (IMT-GT) 2026, yang menempatkan Sumatera Utara sebagai tuan rumah. Forum internasional ini mencakup 35 provinsi dan negara bagian di Sumatera, Semenanjung Malaysia, dan wilayah selatan Thailand. Rangkaian pertemuan berpuncak pada Pertemuan Menteri IMT-GT ke-32 yang dipimpin Menteri Koordinator Bidang Perekonomian RI Airlangga Hartarto. Forum tersebut dihadiri Menteri Ekonomi Malaysia Akmal Nasrullah bin Mohd Nasir dan Vice Minister for the Office of the Prime Minister of Thailand Thanadit Raktabutr. SCROLL TO CONTINU


--- Data 7 ---
ID Berita: 8657101
Judul: Andre Rosiade-Pemprov Sumbar Bakal Temui Warga untuk Dialog Bahas Tol Bukittinggi-Sicincin
Abstrak (Raw): Wakil KetuaKomisi VI DPR RIdari Fraksi Gerindra Andre Rosiade akan turun langsung ke sejumlah jorong di Kabupaten Agam untuk berdialog dengan masyarakat dan ninik mamak terkait rencana pembangunan Jalan Tol Pekanbaru-Padang ruas Bukittinggi-Sicincin. Dialog tersebut dijadwalkan mulai 17 September 2026 dan melibatkan Pemprov Sumatera Barat, Pemkab Agam, Kementerian Pekerjaan Umum (PU) serta PT Hutama Karya. Andre menegaskan pertemuan tersebut bukan sekadar sosialisasi satu arah, melainkan dialog langsung untuk mendengar kekhawatiran warga sekaligus menjelaskan secara terbuka rencana pembangunan dan pembebasan lahan. Ia bahkan mendorong agar dialog dilakukan dari jorong ke jorong sehingga masyarakat yang berada di titik terdampak mendapatkan informasi secara utuh. "Untuk berdiskusi,ndaksosialisasi. Untuk diskusi dan berdialog langsung. Kalau 


--- Data 8 ---
ID Berita: 8657100
Judul: Hampir Rampung, Dua Jembatan Utama di Aceh Siap Beroperasi
Abstrak (Raw): Pemulihan konektivitas di Aceh terus menunjukkan kemajuan. Pembangunan Jembatan Krueng Tingkeum atau Jembatan Kutablang di Kabupaten Bireuen kini memasuki tahap pengaspalan dengan progres fisik mencapai 95 persen dan ditargetkan dapat difungsionalkan mulai 20 September 2026. Jembatan Kutablang merupakan akses jalan nasional yang menghubungkan wilayah timur Aceh menuju Sumatera Utara. Keberadaannya penting untuk mendukung mobilitas masyarakat, distribusi logistik, serta kelancaran aktivitas ekonomi di kawasan tersebut. Perwakilan Tim Satgas Percepatan Rehabilitasi dan Rekonstruksi (PRR) Pascabencana Sumatera Aceh dari Kementerian Dalam Negeri, Imran, mengatakan percepatan pekerjaan terus dilakukan agar jembatan dapat segera digunakan masyarakat. Hal ini disampaikannya saat mengunjungi Banda Aceh, Rabu (9/9). SCROLL TO CONTINUE WITH CONTENT "Kemarin kami sudah berkunjung ke


--- Data 9 ---
ID Berita: 8657093
Judul: Bupati Badung Dorong PSPS Bhakti Negara Cup Jadi Wadah Pembinaan Atlet
Abstrak (Raw): Bupati Badung I Wayan Adi Arnawa mengapresiasi kepada DPC PSPS Bhakti Negara Badung atas penyelenggaraan Kejuaraan Pencak Silat PSPS Bhakti Negara Cup I Tahun 2026. Ia menegaskan kejuaraan ini memegang peran taktis yang lebih besar daripada sekadar berburu podium juara. Ajang regional bertajuk "Berjiwa Pendekar Sejati" ini juga diproyeksikan sebagai instrumen evaluasi komprehensif sekaligus wadah pembinaan atlet secara sistematis dan berkelanjutan."Kejuaraan ini merupakan langkah taktis dalam ruang lingkup keolahragaan. Kita tidak hanya mencari pemenang, tetapi menjadikannya sebagai sarana evaluasi kualitatif terhadap pola latihan yang selama ini diterapkan di masing-masing DPC," ujar Adi Arnawa dalam keterangan tertulis, Kamis (10/9/2026). Hal tersebut disampaikannya saat membuka Kejuaraan Pencak Silat PSPS Bhakti Negara Cup I Tahun 2026 di GOR Purna Krida, K

KeyboardInterrupt: 

In [ ]:
import re
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer

# Gunakan data hasil crawling yang sudah ada di DataFrame.
if "df" not in globals():
    df = pd.read_csv("crawling_detik_berita.csv")

kolom_teks = "isi_berita_original"
if kolom_teks not in df.columns:
    raise KeyError(f"Kolom '{kolom_teks}' tidak ditemukan. Kolom yang tersedia: {list(df.columns)}")

teks_berita = df[kolom_teks].fillna("").astype(str)

# Ambil kata alfabet, ubah menjadi huruf kecil, dan abaikan kata satu huruf.
vectorizer = CountVectorizer(
    lowercase=True,
    token_pattern=r"(?u)\b[a-zA-ZÀ-ÿ]{2,}\b"
)
matriks_kata = vectorizer.fit_transform(teks_berita)
kata_unik = vectorizer.get_feature_names_out()

print(f"Jumlah berita yang dianalisis: {len(teks_berita)}")
print(f"Jumlah kata unik: {len(kata_unik)}")
print("10 kata unik pertama:", list(kata_unik[:10]))

['baru', 'belajar', 'dan', 'hal', 'itu', 'menyenangkan', 'mudah', 'pemrograman', 'python', 'sangat', 'saya', 'suka']
